# 3.2 - Writing to Tables

This notebook demonstrates techniques for creating, updating, and merging tables from file sources, performing incremental data processing, and leveraging Databricks Delta features for efficient data management.

In [0]:
%sql
-- Set catalog and database context for this session
-- Ensures all downstream SQL operates in the intended schema
USE CATALOG databricks_demo;
USE SCHEMA default;

In [0]:
%sql
-- Load order data from CSV into Delta table
-- Demonstrates CREATE TABLE AS SELECT pattern for ingesting file data
CREATE OR REPLACE TABLE tb_orders
AS
SELECT *
FROM csv.`/Volumes/databricks_demo/default/files_data/orders.csv`
WITH (header = "true", inferSchema = "true");

In [0]:
%sql
-- Preview first 3 records from tb_orders
-- Useful for verifying table creation and basic content after load
SELECT * FROM tb_orders limit 3;

In [0]:
%sql
-- Show transaction history for tb_orders table
-- DESCRIBE HISTORY reveals Delta change log, data lineage, and audit trail

In [0]:
%sql
-- Count rows in tb_orders table
-- Validates file ingest, data completeness, and any overwrites

In [0]:
%sql
-- Overwrite tb_orders table by reloading from file
-- Shows INSERT OVERWRITE pattern, useful for resetting data
INSERT OVERWRITE tb_orders
SELECT * FROM csv.`/Volumes/databricks_demo/default/files_data/orders.csv`
WITH (header = "true");

In [0]:
%sql
-- Recount rows after overwrite operation
-- Checks for data consistency and confirms overwrite was successful
SELECT count(*) FROM tb_orders;

In [0]:
%sql
-- View transaction history post-overwrite
-- Use for auditing data changes and tracking table evolution
DESCRIBE HISTORY tb_orders;

In [0]:
%sql
-- Load incremental customer update file as a temp view
-- Execute a MERGE to upsert customer changes into tb_customers
-- Demonstrates Delta's support for complex incrementals (update, insert)
CREATE OR REPLACE TEMP VIEW vw_cutomer_updates AS
SELECT *
FROM csv.`/Volumes/databricks_demo/default/files_data/customer_updates.csv`
WITH (header = "true", inferSchema = "true");
MERGE INTO tb_customers tc
USING vw_cutomer_updates vcu
ON tc.customer_id = vcu.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
-- Load book update file as temp view and upsert changes into tb_books
-- Shows schema declaration for temp view, file options, and Delta MERGE for books
CREATE OR REPLACE TEMP VIEW vw_books_update
  (book_id STRING, title STRING, author STRING, category STRING, price DOUBLE)
USING CSV
OPTIONS (path => "/Volumes/databricks_demo/default/files_data/books_update.csv",
        header = "true", inferSchema = "true");
MERGE INTO tb_books tb
USING vw_books_update vbu
ON tb.book_id = vbu.book_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
-- Preview merged books data
SELECT * FROM tb_books;

## **Incremental Data Process Pipeline Test**
-- Section heading for pipeline operations, organizing following cells by purpose

In [0]:
%sql
-- Create Initial Books Table
CREATE TABLE IF NOT EXISTS tb_books_data (
    book_id STRING,
    title STRING,
    author STRING,
    category STRING,
    price INT
)
USING DELTA;

INSERT INTO tb_books_data VALUES
('B14','Data Communications and Networking','Behrouz A. Forouzan','Computer Science',34),
('B15','Inside the Java Virtual Machine','Bill Venners','Computer Science',41),
('B13','Linux Pocket Guide','Daniel J. Barrett','Computer Science',26),
('B16','Green for Life','Victoria Boutenko','Food',18),
('B17','Cooking with Love','Carla Hall','Food',23);

In [0]:
%sql
-- Preview initial books data after creation and inserts
-- Ensures schema and contents match expectations
SELECT * FROM tb_books_data;

In [0]:
%sql
-- Perform another batch insert into books table
-- Demonstrates append-only and idempotency using SQL
INSERT INTO tb_books_data VALUES
('B14','Data Communications and Networking','Behrouz A. Forouzan','Computer Science',34),
('B15','Inside the Java Virtual Machine','Bill Venners','Computer Science',41),
('B13','Linux Pocket Guide','Daniel J. Barrett','Computer Science',26),
('B16','Green for Life','Victoria Boutenko','Food',18),
('B17','Cooking with Love','Carla Hall','Food',23);

In [0]:
%sql
-- Preview books data to observe effect of repeated inserts
-- Check for duplicate rows and data state
SELECT * FROM tb_books_data;

In [0]:
%sql
-- Insert new records (no updates, no duplicates)
INSERT INTO tb_books_data VALUES
('B18','Deep Learning Basics','Ian Goodfellow','AI',55),
('B19','Data Engineering Guide','Joe Reis','Data Engineering',60);

In [0]:
%sql
-- Update price for a specific record (book_id = 'B13')
-- Demonstrates in-place update with SQL UPDATE
UPDATE tb_books_data
SET price = 30
WHERE book_id = 'B13';

In [0]:
%sql
-- Delete a record by book_id (book_id = 'B16')
-- Demonstrates in-place delete with SQL DELETE
DELETE FROM tb_books_data
WHERE book_id = 'B16';

In [0]:
%sql
-- Prepare batch upserts: update + insert using temp view
-- Useful for batch incremental changes to Delta tables
CREATE OR REPLACE TEMP VIEW books_updates AS
SELECT * FROM VALUES
('B14','Data Communications and Networking','Behrouz A. Forouzan','Computer Science',38), -- update
('B20','Spark in Action','Jean-Georges Perrin','Big Data',50) -- insert
AS t(book_id, title, author, category, price);

In [0]:
%sql
-- Execute MERGE for UPSERT (batch insert + update)
-- Uses changes from temp view to modify books_data
MERGE INTO tb_books_data AS target
USING books_updates AS source
ON target.book_id = source.book_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
-- Prepare batch changes for full MERGE (update, insert, delete)
CREATE OR REPLACE TEMP VIEW books_full_changes AS
SELECT * FROM VALUES
('B14','Data Communications and Networking','Behrouz A. Forouzan','Computer Science',40), -- update
('B21','AI for Beginners','Andrew Ng','AI',45) -- insert
AS t(book_id, title, author, category, price);

In [0]:
%sql
-- Full MERGE statement combining update, insert, and delete
-- Advanced merge: WHEN MATCHED (update), NOT MATCHED (insert), NOT MATCHED BY SOURCE (delete)
MERGE INTO tb_books_data AS target
USING books_full_changes AS source
ON target.book_id = source.book_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE;

In [0]:
%sql
-- Preview final books_data after all merges and pipeline operations
SELECT * FROM tb_books_data;